In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)

device = torch.device('cuda' if torch.cuda.is_available() 
         else 'mps' if torch.backends.mps.is_available() 
         else 'cpu')
device

device(type='cuda')

In [2]:
with open('../data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
len(text)

1765486

In [3]:
vocab = sorted(list(set(text)))
vocab_size = len(vocab)
vocab_size

46

In [4]:
stoi = {ch:i for i, ch in enumerate(vocab)}
itos = {i:ch for i, ch in enumerate(vocab)}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: ''.join([itos[i] for i in l])
decode(encode('hi there'))

'hi there'

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
n = (int)(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]
len(data) == len(train_data) + len(val_data)

True

In [6]:
block_size = 256
batch_size = 64
max_iterations = 5000
eval_iterations = 200
eval_interval = 500
learning_rate = 3e-4
n_embed = 384
n_head = 6
n_layers = 6
dropout = 0.2

In [7]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    idx = torch.randint(len(data)-block_size, (batch_size,))
    xb = torch.stack([data[i:i+block_size] for i in idx])
    yb = torch.stack([data[i+1:i+block_size+1] for i in idx])
    x, y = xb.to(device), yb.to(device)
    return x,y

In [8]:
x, y = get_batch('train')
x, y

(tensor([[ 1, 34, 16,  ..., 29, 20, 15],
         [15, 16, 15,  ..., 26, 30, 31],
         [26, 15, 16,  ...,  1, 25, 26],
         ...,
         [26, 31, 31,  ..., 34, 12, 23],
         [16, 29, 30,  ..., 16, 12, 24],
         [18, 19,  0,  ..., 31, 19,  6]], device='cuda:0'),
 tensor([[34, 16, 15,  ..., 20, 15, 16],
         [16, 15,  2,  ..., 30, 31,  1],
         [15, 16,  0,  ..., 25, 26,  1],
         ...,
         [31, 31, 16,  ..., 12, 23, 22],
         [29, 30,  1,  ..., 12, 24, 30],
         [19,  0, 34,  ..., 19,  6,  1]], device='cuda:0'))

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        dim_head = n_embed // num_heads
        self.multiheads = nn.ModuleList([SingleHeadAttention(dim_head) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        out = torch.cat([h(x) for h in self.multiheads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [10]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.ffw = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),   
            nn.Linear(4 * n_embed, n_embed), 
            nn.Dropout(dropout)        
        )

    def forward(self, x):
        out = self.ffw(x)
        return out

In [11]:
class Block(nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        self.ffwd = FeedForward()
        self.ma = MultiHeadAttention(num_heads)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.ma(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [12]:
class SingleHeadAttention(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False) #E,H
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        B,T,E = x.shape
        k = self.key(x) #B,T,E @ E,H -> B,T,H
        q = self.query(x)
        v = self.value(x)
        
        wei = k @ q.transpose(-2, -1) * E ** -0.5 #B,T,H @ B,H,T -> B,T,T
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        wei = wei.softmax(dim=-1)       
        wei = self.dropout(wei)
         
        out = wei @ v #B,T,T @ B,T,H -> B,T,H
        return out

In [13]:
class Bitsy(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.pos_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*[Block(n_head) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape
        tok_emb = self.token_embedding_table(idx) #BTV
        pos_emb = self.pos_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x) #BTH == BTE
        x = self.ln(x)
        logits = self.lm_head(x) #BTE @ EV => BTV
        
        if targets == None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)        
        return logits, loss

    @torch.no_grad()
    def evaluate(self):
        out = {}
        self.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iterations)
            for k in range(eval_iterations):
                x, y = get_batch(split)
                logits, loss = self(x,y)
                losses[k] = loss.item()
            out[split] = losses.mean()
        self.train()
        return out

    @torch.no_grad()
    def generate(self, idx, max_tokens=300):
        for _ in range(max_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=-1)
        return idx

In [14]:
model = Bitsy()
m = model.to(device)
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)
m


Bitsy(
  (token_embedding_table): Embedding(46, 384)
  (pos_embedding_table): Embedding(256, 384)
  (blocks): Sequential(
    (0): Block(
      (ffwd): FeedForward(
        (ffw): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ma): MultiHeadAttention(
        (multiheads): ModuleList(
          (0-5): 6 x SingleHeadAttention(
            (key): Linear(in_features=384, out_features=64, bias=False)
            (query): Linear(in_features=384, out_features=64, bias=False)
            (value): Linear(in_features=384, out_features=64, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_af

In [15]:

#training loop
for k in range(max_iterations):
    if k % eval_interval == 0:
        losses = model.evaluate()
        print(f"step: {k}, training_loss: {losses['train']:.4f}, val_loss: {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(f"step: {k}, training_loss: {losses['train']:.4f}, val_loss: {losses['val']:.4f}")


step: 0, training_loss: 3.8891, val_loss: 3.8895
step: 500, training_loss: 2.0396, val_loss: 2.0479
step: 1000, training_loss: 1.7275, val_loss: 1.7307
step: 1500, training_loss: 1.5939, val_loss: 1.6096
step: 2000, training_loss: 1.5111, val_loss: 1.5422
step: 2500, training_loss: 1.4545, val_loss: 1.4935
step: 3000, training_loss: 1.4056, val_loss: 1.4631
step: 3500, training_loss: 1.3675, val_loss: 1.4420
step: 4000, training_loss: 1.3371, val_loss: 1.4264
step: 4500, training_loss: 1.3044, val_loss: 1.4124
step: 4999, training_loss: 1.3044, val_loss: 1.4124


In [18]:
idx = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(idx, max_tokens=1000)[0].tolist()))


the searchest never start into the roar,
but never bested the verse,
and it such hurroun from the fagher reference,
the peasure scrowd it begun to pour as dower
by the tear -they gave them goth, as at more,
togeting all with a distant; as
have reft a woman: here was slain
they wourn'd the worst drappine of force
the sea-town, ode for the plopint driven crava
but more than pispiouses ocean away!
who drenden and woman days of sage tide,
and love along him the midain'd depth or ever,
swarn, wh'n one find of desk that throesay:
he stopp'd not to war a brote whelm?
bled monstrain his eye eye known --
     
that addid therefstance they play'd his mast!
 
and himself, -- not onward's rolly done,
take menetees, while his now wreck'd power
the waters, in wealth, bequieted wrought
i've quite recalled by things whose trade,
and clad on their effrit kissing to his faces;
which all decove the life crowd the countless of -
a port of senty place sistened,
who hath that loss, but daring and pity
for 